## Convolutional Neural Networks

`keras` directly provides layers for building convolutional networks, such as `Conv2D` for a two-dimensional convolution layer and `MaxPooling2D` for *pooling* with a maximum aggregation function.

In [ ]:
# import required libraries
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(12345)

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

# `keras` directly provides several demo datasets; MNIST is a dataset
# of images of handwritten digits from 0 to 9
from tensorflow.keras.datasets import mnist
from tensorflow.keras import utils

### Data preprocessing

During preprocessing, we load the data into `numpy` arrays and normalize their values.

In [ ]:
# load training and testing data
(x_train, y_train), (x_test, y_test) = mnist.load_data()
# we have 60000 training grayscale images with dimensions 28x28 pixels
# and 10000 testing images
print(x_train.shape, x_test.shape)

In [ ]:
# we will display the first training example as an image
plt.imshow(x_train[0], cmap='gray')

In [ ]:
# we need to reshape the data into a tensor with dimensions
# number of images x width x height x number of color channels
# since we have grayscale images, we only have one color channel
x_train = x_train.reshape(x_train.shape[0], 28, 28, 1)
x_test = x_test.reshape(x_test.shape[0], 28, 28, 1)

# original values are integers from 0 (black) to 255 (white)
# we will change the type to floating-point numbers and normalize the data to the range 0-1
x_train = x_train.astype('float32')
x_test = x_test.astype('float32')
x_train /= 255
x_test /= 255

In [ ]:
# the target categorical attribute needs to be encoded for the output layer as a binary vector
# for binary encoding you can use a helper function from the `keras` library
y_train = utils.to_categorical(y_train, 10)
y_test = utils.to_categorical(y_test, 10)

### Model creation

In [ ]:
# we create a sequential model
model = Sequential()

# 32 filters, size 3x3, input image 28x28 pixels x 1 color channel
model.add(Conv2D(32, kernel_size=(3,3), activation='relu', input_shape=(28,28,1)))
# 32 filters, size 3x3
model.add(Conv2D(32, kernel_size=(3,3), activation='relu'))
# pooling, size 2x2
model.add(MaxPooling2D(pool_size=(2,2)))
# dropout for regularization during training
model.add(Dropout(0.25))

# Flatten transforms the input from a two-dimensional grid into a one-dimensional vector for
# the feedforward network
model.add(Flatten())
# feedforward network with one hidden layer - 128 neurons
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))
# output layer for 10 classes
model.add(Dense(10, activation='softmax'))

# we display the model structure and the number of parameters in each layer,
# note that most parameters are in the feedforward network
model.summary()

In [ ]:
# you can also visualize the model graphically
# to display the model you need to have the `graphviz` program installed
# utils.plot_model(model, to_file='convolution_model.png', show_shapes=True)

### Training and evaluation of accuracy

In [ ]:
adam = Adam()
# standard training procedure for classification
model.compile(loss='categorical_crossentropy', optimizer=adam, metrics=['accuracy'])

In [ ]:
f = model.fit(x_train, y_train, epochs=3, batch_size=128, validation_data=(x_test, y_test))

In [ ]:
# we will display the first test example
plt.imshow(x_test[0].reshape(28,28), cmap='gray')

In [ ]:
predictions = model.predict(x_test[:1])
# we will display the prediction for individual classes (digits) for the first test example
print(predictions[0])
plt.bar(np.arange(10), predictions[0])
plt.xticks(np.arange(10))